In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a GPU program that performs Top-K Gating for Mixture of Experts (MoE) models. Given a logit matrix of shape <code>[M, E]</code> where M is the number of tokens and E is the number of experts, identify the k largest values in each row, extract their indices, and apply softmax to get mixing weights.
</p>

<p>
  For each row i, the operation computes:
  $$
  \begin{align}
  \text{indices}_i, \text{vals}_i &= \text{TopK}(\text{logits}_i, k) \\
  \text{vals}_i &= \text{logits}_i[\text{indices}_i] \\
  \text{weights}_i &= \text{Softmax}(\text{vals}_i)
  \end{align}
  $$
</p>

<p>
  The selected experts must remain ordered by descending logit value, matching the order returned by
  <code>topk</code>. The <code>topk_weights</code> array must correspond positionally to
  <code>topk_indices</code> in that same order.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>External libraries are not permitted</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the <code>topk_weights</code> and <code>topk_indices</code> arrays</li>
</ul>

<h2>Example 1:</h2>
<pre>
Input:
  logits = [[1.0, 2.0, 3.0, 4.0],
            [4.0, 3.0, 2.0, 1.0]]
  M = 2, E = 4, k = 2

Output:
  topk_weights = [[0.7311, 0.2689],
                  [0.7311, 0.2689]]
  topk_indices = [[3, 2],
                  [0, 1]]

Explanation:
Row 0: Top-2 values are 4.0 and 3.0 at indices 3 and 2.
       Softmax([4.0, 3.0]) = [0.7311, 0.2689]
Row 1: Top-2 values are 4.0 and 3.0 at indices 0 and 1.
       Softmax([4.0, 3.0]) = [0.7311, 0.2689]
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 ≤ <code>M</code> ≤ 10,000 (number of tokens)</li>
  <li>1 ≤ <code>E</code> ≤ 256 (number of experts)</li>
  <li>1 ≤ <code>k</code> ≤ <code>E</code> (top-k selection, typically k=2)</li>
  <li>All tensors are stored on GPU</li>
  <li>Logits are 32-bit floats</li>
  <li>Indices are 32-bit integers</li>

  <li>Performance is measured with <code>M</code> = 1,024, <code>k</code> = 2</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// logits, topk_weights, topk_indices are device pointers
extern "C" void solve(const float* logits, float* topk_weights, int* topk_indices, int M, int E,
                      int k) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# logits, topk_weights, topk_indices are tensors on the GPU
@cute.jit
def solve(
    logits: cute.Tensor,
    topk_weights: cute.Tensor,
    topk_indices: cute.Tensor,
    M: cute.Int32,
    E: cute.Int32,
    k: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# logits is a tensor on the GPU
@jax.jit
def solve(logits: jax.Array, M: int, E: int, k: int) -> tuple[jax.Array, jax.Array]:
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


@export
def solve(
    logits: UnsafePointer[Float32, MutExternalOrigin],
    topk_weights: UnsafePointer[Float32, MutExternalOrigin],
    topk_indices: UnsafePointer[Int32, MutExternalOrigin],
    M: Int32,
    E: Int32,
    k: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# logits, topk_weights, topk_indices are tensors on the GPU
def solve(
    logits: torch.Tensor,
    topk_weights: torch.Tensor,
    topk_indices: torch.Tensor,
    M: int,
    E: int,
    k: int,
):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# logits, topk_weights, topk_indices are tensors on the GPU
def solve(
    logits: torch.Tensor,
    topk_weights: torch.Tensor,
    topk_indices: torch.Tensor,
    M: int,
    E: int,
    k: int,
):
    pass


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/medium/67_moe_topk_gating/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
